RAG Pipelines- Data Ingestion to Vector DB pipeline

In [5]:
import os
from pathlib import Path
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12048\1850894251.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [45]:
def process_pdf(pdf_directory):
    """process all pdf files in a directory and return a list of documents"""
    documents=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} pdf files in to process....")
    for pdf_file in pdf_files:
        print(f"\n processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            pdf_documents = loader.load()
            for doc in pdf_documents:
                doc.metadata["source"] = str(pdf_file.name)
                doc.metadata["file_type"] = 'pdf'
            documents.extend(pdf_documents)
            print(f"✓ loaded {len(pdf_documents)} pages from {pdf_file.name}")
        except Exception as e:
            print(f"✕ Error loading {pdf_file.name}: {e}")
            continue
    print(f"loaded a total of {len(documents)} documents from {len(pdf_files)} pdf files")    
    return documents
all_documents = process_pdf("../data/pdfs")

found 7 pdf files in to process....

 processing Abhinav_Thakur_Resume 2 1.pdf


✓ loaded 1 pages from Abhinav_Thakur_Resume 2 1.pdf

 processing Computer Networks Week-3 Assignment.docx.pdf
✓ loaded 2 pages from Computer Networks Week-3 Assignment.docx.pdf

 processing DevMind_AI_Synopsis.pdf
✓ loaded 9 pages from DevMind_AI_Synopsis.pdf

 processing Lecture 22a.pdf
✓ loaded 13 pages from Lecture 22a.pdf

 processing Lecture 23b.pdf
✓ loaded 9 pages from Lecture 23b.pdf

 processing Lecture 24a.pdf
✓ loaded 10 pages from Lecture 24a.pdf

 processing Week-10.pdf
✓ loaded 121 pages from Week-10.pdf
loaded a total of 165 documents from 7 pdf files


In [46]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]   
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print("\nExample of a chunk:")
        print(f"first chunk: {split_docs[0].page_content[:200]}...")
        print(f"metadata: {split_docs[0].metadata}")
    return split_docs

In [47]:
chunks = split_documents(all_documents, chunk_size=1000, chunk_overlap=200)

split 165 documents into 177 chunks

Example of a chunk:
first chunk: ABHINAV THAKUR 
New Delhi, India 110037 | +91 7428203457 |  abhinav2005thakur@gmail.com 
  Github | Linkedin | Portfolio 
PROFESSIONAL SUMMARY 
Full-Stack Developer who builds complete web application...
metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2026-10-30T04:35:01-07:00', 'author': 'Un-named', 'moddate': '2026-10-30T04:35:01-07:00', 'source': 'Abhinav_Thakur_Resume 2 1.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}


Now EMbedding from here

In [37]:
%pip install requirements.txt

HINT: You are attempting to install a package literally named "requirements.txt" (which cannot exist). Consider using the '-r' flag to install the packages listed in requirements.txt
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement requirements.txt (from versions: none)
ERROR: No matching distribution found for requirements.txt


In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [2]:
%pip install langchain langchain-community langchain-core langchain-text-splitters langchain-google-genai langchain-nvidia-ai-endpoints langchain-chroma chromadb pypdf python-dotenv sentence-transformers transformers torch numpy pandas scikit-learn

  Using cached langchain_google_genai-4.4.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_nvidia_ai_endpoints-1.5.0-py3-none-any.whl.metadata (23 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached sentence_transformers-6.2.0-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.22.1-py3-none-any.whl.metadata (31 kB)
  Using cached torch-2.15.0-cp312-cp312-win_amd64.whl.metadata (36 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached google_genai-2.34.0-py3-none-any.whl.metadata (59 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pybase64-1.5.1-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.2-p

In [48]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [49]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}...")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Loaded model: {self.model_name}, "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        if self.model is None:
            raise ValueError("Embedding model is not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings
embedding_manager= EmbeddingManager(model_name="all-MiniLM-L6-v2")
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 704.98it/s]


Loaded model: all-MiniLM-L6-v2, Embedding dimension: 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12048\854918505.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


In [50]:
texts = [
    "Machine learning is a subset of AI.",
    "Python is a programming language."
]

embeddings = embedding_manager.generate_embeddings(texts)

print(embeddings.shape)

Generating embeddings for 2 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.87it/s]

Generated embeddings with shape: (2, 384)
(2, 384)


###Vector Store from here

In [57]:

import os
import uuid
import chromadb
import numpy as np

from typing import List, Any


class VectorStore:

    def __init__(
        self,
        collection_name: str = "pdf_embeddings",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF embeddings collection for RAG application"
                }
            )

            print(
                f"Initialized ChromaDB client and collection: "
                f"{self.collection_name}"
            )

            print(
                f"Existing documents: {self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the collection.
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents and embeddings must be the same."
            )

        print(f"Adding {len(documents)} documents...")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):

            doc_id = str(uuid.uuid4())
            ids.append(doc_id)

            metadata = {}
            metadata.update(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            documents_texts.append(doc.page_content)

            embeddings_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )

            print(
                f"Successfully added {len(documents)} documents."
            )

            print(
                f"Collection now contains {self.collection.count()} documents."
            )

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


vstore = VectorStore(
    collection_name="pdf_embeddings",
    persist_directory="../data/vector_store"
)

vstore


Initialized ChromaDB client and collection: pdf_embeddings
Existing documents: 708


In [58]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2026-10-30T04:35:01-07:00', 'author': 'Un-named', 'moddate': '2026-10-30T04:35:01-07:00', 'source': 'Abhinav_Thakur_Resume 2 1.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='ABHINAV THAKUR \nNew Delhi, India 110037 | +91 7428203457 |  abhinav2005thakur@gmail.com \n  Github | Linkedin | Portfolio \nPROFESSIONAL SUMMARY \nFull-Stack Developer who builds complete web applications, from the user interface to the backend and database. Built and \ndeployed projects including an AI tool that answers questions about code and a security tool that blocks common web attacks and \nworked as a MERN Stack Intern. Springer-published researcher with strong problem-solving and DSA skills. \nTECHNICAL SKILLS \nLanguages & Databases C++, Python, JavaScript, TypeScript, SQL, PostgreSQL, MongoDB \nFrontend      React.js, Tailwind CSS, HTML, CSS, react-router-dom, React Hooks \nBa

In [59]:
texts=[doc.page_content for doc in chunks]
texts

['ABHINAV THAKUR \nNew Delhi, India 110037 | +91 7428203457 |  abhinav2005thakur@gmail.com \n  Github | Linkedin | Portfolio \nPROFESSIONAL SUMMARY \nFull-Stack Developer who builds complete web applications, from the user interface to the backend and database. Built and \ndeployed projects including an AI tool that answers questions about code and a security tool that blocks common web attacks and \nworked as a MERN Stack Intern. Springer-published researcher with strong problem-solving and DSA skills. \nTECHNICAL SKILLS \nLanguages & Databases C++, Python, JavaScript, TypeScript, SQL, PostgreSQL, MongoDB \nFrontend      React.js, Tailwind CSS, HTML, CSS, react-router-dom, React Hooks \nBackend     Node.js, Express.js, RESTful API Design, Socket.io, Middleware \nAuth & Security     JWT, GitHub OAuth \nAI & Data     RAG, AI API Integration, yfinance \nTools                                           Git, GitHub, Render \nCore     Data Structures & Algorithms, Object-Oriented Programming

In [60]:
embeddings=embedding_manager.generate_embeddings(texts)

Generating embeddings for 177 texts...


Batches: 100%|██████████| 6/6 [00:12<00:00,  2.06s/it]

Generated embeddings with shape: (177, 384)


In [61]:
VectorStore.add_documents(vstore, chunks, embeddings)

Adding 177 documents...
Successfully added 177 documents.
Collection now contains 885 documents.


RAG  retrieval store

In [62]:

def retrieve(
    self,
    query: str,
    top_k: int = 5,
    distance_threshold: float = 1.7
) -> List[Dict[str, Any]]:

    print(f"Retrieving documents for: {query}")

    query_embedding = self.embedding_manager.generate_embeddings([query])[0]

    try:
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["metadatas", "documents", "distances"]
        )

        retrieved_docs = []

        if not results["documents"] or not results["documents"][0]:
            print("No documents found.")
            return []

        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]
        ids = results["ids"][0]

        for i, (doc_id, doc, meta, dist) in enumerate(
            zip(ids, documents, metadatas, distances)
        ):

            if dist <= distance_threshold:
                retrieved_docs.append({
                    "id": doc_id,
                    "content": doc,
                    "metadata": meta,
                    "distance": dist,
                    "rank": i + 1
                })

        print(f"Retrieved {len(retrieved_docs)} documents.")

        return retrieved_docs

    except Exception as e:
        print(f"Error during retrieval: {e}")
        return []

In [64]:
rag_retriever = RAGretriever(
    vector_store=vstore,
    embedding_manager=embedding_manager
)

In [65]:
rag_retriever.retrieve(
    query="Scheduling policies: priority",
    top_k=5,
    score_threshold=-1.0
)

Retrieving documents for query: 'Scheduling policies: priority' with top_k=5 and score_threshold=-1.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.01it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents after applying score threshold.


[{'id': 'c61b3edb-3421-4e70-8079-709bee8ed8ef',
  'content': '❑Priority scheduling: \n❑ arriving traffic classified, \nqueued by class\n❑ any header fields can be \nused for classification\nScheduling policies: priority\nhigh priority queue\nlow priority queue\narrivals\nclassify departureslink\n1 3 2 4 5\narrivals\ndepartures\npacket \nin \nservice\n❑ send packet from highest \npriority queue that has \nbuffered packets\n❑ FCFS within priority class\n1 3 4\n2\n5\n1 3 2 4 5\nIIT Indore  © Neminah Hubballi 3',
  'metadata': {'source': 'Lecture 23b.pdf',
   'creationdate': '2026-07-29T08:29:18+00:00',
   'page': 2,
   'author': 'Dr. Neminath Hubballi',
   'file_name': 'pdf',
   'producer': 'Adobe PDF Services',
   'moddate': '2026-07-29T15:41:25+05:30',
   'page_label': '3',
   'total_pages': 9,
   'creator': 'Microsoft® PowerPoint® 2019',
   'title': '',
   'content_length': 426,
   'doc_index': 35},
  'score': 0.1540181040763855,
  'distance': 0.8459818959236145,
  'rank': 1},
 {'id': 

In [66]:
query = "What is lecture 22a"

query_embedding = embedding_manager.generate_embeddings([query])[0]

results = vstore.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

print("Documents:", results["documents"])
print("Distances:", results["distances"])
print("Metadata:", results["metadatas"])

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.59it/s]

Generated embeddings with shape: (1, 384)
Documents: [['Variable Sized Networks\n❑ Classless Inter Domain Routing (CIDR): \n❑ Subnet portion of address of arbitrary length\n❑ Address format: a.b.c.d/x, where x is # bits in subnet \nportion of address\n11001000  00010111  00010000  00000000\nsubnet\npart\nhost\npart\n200.23.16.0/23\nIIT Indore©Neminath Hubballi', 'Variable Sized Networks\n❑ Classless Inter Domain Routing (CIDR): \n❑ Subnet portion of address of arbitrary length\n❑ Address format: a.b.c.d/x, where x is # bits in subnet \nportion of address\n11001000  00010111  00010000  00000000\nsubnet\npart\nhost\npart\n200.23.16.0/23\nIIT Indore©Neminath Hubballi', 'Variable Sized Networks\n❑ Classless Inter Domain Routing (CIDR): \n❑ Subnet portion of address of arbitrary length\n❑ Address format: a.b.c.d/x, where x is # bits in subnet \nportion of address\n11001000  00010111  00010000  00000000\nsubnet\npart\nhost\npart\n200.23.16.0/23\nIIT Indore©Neminath Hubballi', 'Variable Sized

In [67]:
print("Documents:", results["documents"])
print("Distances:", results["distances"])
print("Metadata:", results["metadatas"])

Documents: [['Variable Sized Networks\n❑ Classless Inter Domain Routing (CIDR): \n❑ Subnet portion of address of arbitrary length\n❑ Address format: a.b.c.d/x, where x is # bits in subnet \nportion of address\n11001000  00010111  00010000  00000000\nsubnet\npart\nhost\npart\n200.23.16.0/23\nIIT Indore©Neminath Hubballi', 'Variable Sized Networks\n❑ Classless Inter Domain Routing (CIDR): \n❑ Subnet portion of address of arbitrary length\n❑ Address format: a.b.c.d/x, where x is # bits in subnet \nportion of address\n11001000  00010111  00010000  00000000\nsubnet\npart\nhost\npart\n200.23.16.0/23\nIIT Indore©Neminath Hubballi', 'Variable Sized Networks\n❑ Classless Inter Domain Routing (CIDR): \n❑ Subnet portion of address of arbitrary length\n❑ Address format: a.b.c.d/x, where x is # bits in subnet \nportion of address\n11001000  00010111  00010000  00000000\nsubnet\npart\nhost\npart\n200.23.16.0/23\nIIT Indore©Neminath Hubballi', 'Variable Sized Networks\n❑ Classless Inter Domain Routin